In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
import time
import warnings
warnings.filterwarnings('ignore')

from config.paths import config

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
import joblib

processed_data = joblib.load(config.processed_data_dir / "feature_engineered_data.pkl")
X_train = processed_data['X_train']
X_test = processed_data['X_test']
y_train = processed_data['y_train']
y_test = processed_data['y_test']
feature_names = processed_data['feature_names']

print(f"Training data: {X_train.shape}")
print(f"Testing data: {X_test.shape}")
print(f"Feature names: {len(feature_names)}")
print(f"Class distribution - Training: {y_train.value_counts(normalize=True).to_dict()}")

In [ ]:
models = {
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=500, class_weight='balanced', n_jobs=-1),
        'reason': 'Fast baseline interpretable model for linear relationships'
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, n_estimators=50, class_weight='balanced', n_jobs=-1),
        'reason': 'Fast ensemble that handles non-linear relationships and provides feature importance'
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, n_estimators=50, eval_metric='logloss',
                              scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
                              n_jobs=-1),
        'reason': 'Optimized gradient boosting for tabular data'
    },
    'LightGBM': {
        'model': LGBMClassifier(random_state=42, n_estimators=50, class_weight='balanced', n_jobs=-1),
        'reason': 'Very fast gradient boosting optimized for large datasets'
    },
    'Linear SVM': {
        'model': LinearSVC(random_state=42, class_weight='balanced', max_iter=1000),
        'reason': 'Fast linear kernel alternative to standard SVM'
    },
    'KNN': {
        'model': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'reason': 'Fast distance-based approach for local patterns'
    }
}

print("OPTIMIZED MODEL ARSENAL (Fast Training):")
for name, config in models.items():
    print(f"  {name}: {config['reason']}")

In [ ]:
scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("FAST EVALUATION FRAMEWORK:")
print(f"  Cross-validation: {cv_strategy.n_splits}-fold stratified")
print(f"  Metrics: {list(scoring_metrics.keys())}")
print(f"  Expected total time: 2-4 minutes")

In [ ]:
results = {}
training_times = {}

print("FAST CROSS-VALIDATION PERFORMANCE:")
print("=" * 80)

for model_name, model_config in models.items():
    print(f"\nTraining {model_name}...")
    model = model_config['model']

    start_time = time.time()

    try:
        cv_results = cross_validate(
            model, X_train, y_train,
            cv=cv_strategy,
            scoring=scoring_metrics,
            return_train_score=False,
            n_jobs=-1
        )

        training_time = time.time() - start_time
        training_times[model_name] = training_time
        results[model_name] = cv_results

        print(f"  {model_name} - Training time: {training_time:.2f}s")

        for metric in scoring_metrics.keys():
            test_key = f'test_{scoring_metrics[metric]}'
            if test_key in cv_results:
                test_scores = cv_results[test_key]
                print(f"  {metric.upper():10} - Test: {np.mean(test_scores):.4f} ± {np.std(test_scores):.4f}")

    except Exception as e:
        print(f"  {model_name} - Failed: {str(e)}")
        training_times[model_name] = None
        results[model_name] = None

In [ ]:
performance_summary = []

for model_name in models.keys():
    if model_name in results and results[model_name] is not None:
        cv_result = results[model_name]

        model_metrics = {}
        for metric in scoring_metrics.keys():
            test_key = f'test_{scoring_metrics[metric]}'
            if test_key in cv_result:
                test_scores = cv_result[test_key]
                model_metrics[metric] = {
                    'mean': np.mean(test_scores),
                    'std': np.std(test_scores)
                }
            else:
                model_metrics[metric] = {'mean': np.nan, 'std': np.nan}

        performance_summary.append({
            'Model': model_name,
            'Accuracy': f"{model_metrics['accuracy']['mean']:.4f} ± {model_metrics['accuracy']['std']:.4f}",
            'Precision': f"{model_metrics['precision']['mean']:.4f} ± {model_metrics['precision']['std']:.4f}",
            'Recall': f"{model_metrics['recall']['mean']:.4f} ± {model_metrics['recall']['std']:.4f}",
            'F1-Score': f"{model_metrics['f1']['mean']:.4f} ± {model_metrics['f1']['std']:.4f}",
            'ROC-AUC': f"{model_metrics['roc_auc']['mean']:.4f} ± {model_metrics['roc_auc']['std']:.4f}",
            'Training Time (s)': f"{training_times[model_name]:.2f}"
        })

performance_df = pd.DataFrame(performance_summary)
print("\n" + "="*100)
print("FAST BASELINE MODEL PERFORMANCE SUMMARY")
print("="*100)
print(performance_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

for idx, (metric, title) in enumerate(zip(metric_names, metric_titles)):
    model_names = []
    metric_means = []
    metric_stds = []

    for model_name in models.keys():
        if model_name in results and results[model_name] is not None:
            cv_result = results[model_name]
            test_key = f'test_{scoring_metrics[metric]}'
            if test_key in cv_result:
                test_scores = cv_result[test_key]
                model_names.append(model_name)
                metric_means.append(np.mean(test_scores))
                metric_stds.append(np.std(test_scores))

    if model_names:
        bars = axes[idx].bar(model_names, metric_means, yerr=metric_stds, capsize=5, alpha=0.7)
        axes[idx].set_title(f'{title} Comparison', fontsize=14, fontweight='bold')
        axes[idx].set_ylabel(title)
        axes[idx].tick_params(axis='x', rotation=45)

        for bar, mean in zip(bars, metric_means):
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                          f'{mean:.3f}', ha='center', va='bottom', fontsize=9)
    else:
        axes[idx].text(0.5, 0.5, f'No data for {title}', ha='center', va='center', fontsize=12)
        axes[idx].set_title(f'{title} Comparison', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
valid_training_times = {k: v for k, v in training_times.items() if v is not None}
plt.bar(valid_training_times.keys(), valid_training_times.values(), color='lightcoral', alpha=0.7)
plt.title('Fast Model Training Time Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Training Time (seconds)')
plt.xticks(rotation=45)
for i, (model, time_val) in enumerate(valid_training_times.items()):
    plt.text(i, time_val + 0.1, f'{time_val:.1f}s', ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*60)
print("FAST FEATURE IMPORTANCE ANALYSIS")
print("="*60)

rf_model = RandomForestClassifier(random_state=42, n_estimators=50, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train, y_train)

xgb_model = XGBClassifier(random_state=42, n_estimators=50, eval_metric='logloss',
                         scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
                         n_jobs=-1)
xgb_model.fit(X_train, y_train)

rf_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

xgb_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nRANDOM FOREST - Top 10 Feature Importances:")
print(rf_importance.head(10).to_string(index=False))

print("\nXGBOOST - Top 10 Feature Importances:")
print(xgb_importance.head(10).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

rf_top10 = rf_importance.head(10)
axes[0].barh(rf_top10['feature'], rf_top10['importance'], color='skyblue')
axes[0].set_title('Random Forest - Top 10 Feature Importances', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance Score')

xgb_top10 = xgb_importance.head(10)
axes[1].barh(xgb_top10['feature'], xgb_top10['importance'], color='lightgreen')
axes[1].set_title('XGBoost - Top 10 Feature Importances', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

In [ ]:
import json
from config.paths import config  # Re-import to ensure config is correct

# Ensure directories exist
config.create_directories()

baseline_results = {
    'performance_summary': performance_df.to_dict('records'),
    'training_times': training_times,
    'cv_results': {model: {k: v.tolist() if isinstance(v, np.ndarray) else v
                          for k, v in results[model].items()}
                  for model in models.keys() if model in results and results[model] is not None},
    'feature_importance': {
        'random_forest': rf_importance.to_dict('records'),
        'xgboost': xgb_importance.to_dict('records')
    }
}

# Save JSON results
with open(config.reports_dir / 'fast_baseline_models_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

# Train and save models
trained_models = {}
for model_name, model_config in models.items():
    if model_name in results and results[model_name] is not None:
        model = model_config['model']
        model.fit(X_train, y_train)
        trained_models[model_name] = model

joblib.dump(trained_models, config.models_dir / 'fast_baseline_models.pkl')

# Save CSV files
performance_df.to_csv(config.reports_dir / 'fast_model_performance_comparison.csv', index=False)
rf_importance.to_csv(config.reports_dir / 'fast_random_forest_feature_importance.csv', index=False)
xgb_importance.to_csv(config.reports_dir / 'fast_xgboost_feature_importance.csv', index=False)

print("\n" + "="*60)
print("FAST BASELINE MODELING COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"✓ Trained and evaluated {len(trained_models)} optimized models")
print(f"✓ Completed in minutes instead of hours")
print(f"✓ Performed 3-fold cross-validation on all models")
print(f"✓ Analyzed feature importance for top 2 models")
print(f"✓ Saved all results to: {config.reports_dir}")
print(f"✓ Saved trained models to: {config.models_dir}")
print(f"✓ Ready for hyperparameter tuning on best performers")

In [ ]:
print("\n" + "="*80)
print("MODEL SELECTION ANALYSIS")
print("="*80)

# Convert performance metrics to numeric for ranking
performance_ranking = performance_df.copy()

# Extract numeric values from the string format
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    performance_ranking[metric + '_mean'] = performance_ranking[metric].str.extract(r'([\d.]+)').astype(float)
    performance_ranking['Training_Time'] = performance_ranking['Training Time (s)'].str.extract(r'([\d.]+)').astype(float)

# Calculate overall score (weighted average)
weights = {
    'Accuracy_mean': 0.15,
    'Precision_mean': 0.15,
    'Recall_mean': 0.15,
    'F1-Score_mean': 0.20,  # Higher weight for balanced metric
    'ROC-AUC_mean': 0.35    # Highest weight for ranking capability
}

# Calculate weighted score
performance_ranking['Overall_Score'] = (
    performance_ranking['Accuracy_mean'] * weights['Accuracy_mean'] +
    performance_ranking['Precision_mean'] * weights['Precision_mean'] +
    performance_ranking['Recall_mean'] * weights['Recall_mean'] +
    performance_ranking['F1-Score_mean'] * weights['F1-Score_mean'] +
    performance_ranking['ROC-AUC_mean'] * weights['ROC-AUC_mean']
)

# Rank models
performance_ranking['Rank'] = performance_ranking['Overall_Score'].rank(ascending=False)
performance_ranking = performance_ranking.sort_values('Rank')

print("\nMODEL RANKING BY OVERALL PERFORMANCE:")
print("="*70)
ranking_display = performance_ranking[['Model', 'Overall_Score', 'Rank', 'Accuracy', 'F1-Score', 'ROC-AUC', 'Training Time (s)']].copy()
ranking_display['Overall_Score'] = ranking_display['Overall_Score'].round(4)
print(ranking_display.to_string(index=False))

# Identify best model
best_model_name = performance_ranking.iloc[0]['Model']
best_model_score = performance_ranking.iloc[0]['Overall_Score']

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Overall Score: {best_model_score:.4f}")
print(f"   Key Metrics:")
print(f"     - Accuracy: {performance_ranking.iloc[0]['Accuracy']}")
print(f"     - F1-Score: {performance_ranking.iloc[0]['F1-Score']}")
print(f"     - ROC-AUC: {performance_ranking.iloc[0]['ROC-AUC']}")
print(f"     - Training Time: {performance_ranking.iloc[0]['Training Time (s)']}")